# AGOL Demo | Load One Module File from GitHub

This notebook shows a simple pattern for loading one Python module file directly from a GitHub repository into an AGOL notebook session.

**What the helper function `load_github_module` does:**

1. Builds a raw GitHub URL for one `.py` file.
2. Downloads the file to a local cache in `/arcgis/home/modules_cache`.
3. Imports the file dynamically as a Python module.
4. Returns the imported module so you can call its functions.

*Why use this approach?*

- You can use one module file without installing the full package.
- You can pin `reference` to a tag or commit for reproducible scheduled runs.
- You can use `main` for fast development iteration.

In [ ]:
import os
import getpass
from pathlib import Path

from dotenv import load_dotenv
from gis_utils_public import arcgis_utils

### ArcGIS Online Login

You can log in to ArcGIS Online in three ways:

- **Environment variables**: 
  - add credentials to a `.env` file (see `notebooks/.env.example`), then call `get_agol_connection(use_pro=False)`.
- **ArcGIS Pro**: sign in through ArcGIS Pro, then call `get_agol_connection(use_pro=True)`.
- **Terminal input**: if the options above are not used or fail, you can enter username and password manually.

In [ ]:
# --- Load environment variables from .env file ---
env_candidates = [
    Path("notebooks") / ".env",
    Path.cwd() / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)

if env_file is not None:
    load_dotenv(env_file)
    print(f"Loaded variables from {env_file.resolve()}")
else:
    print("No .env file found. Set environment variables or use interactive prompt.")

In [ ]:
# --- Connect to ArcGIS Online ---
def get_agol_connection(use_pro: bool = False):
    """
    Return an authenticated GIS connection to ArcGIS Online.

    :param use_pro: If True, try ArcGIS Pro credentials first.
    :return: Authenticated GIS object.
    """

    from arcgis.gis import GIS

    if use_pro:
        try:
            gis = GIS("home")
            user = gis.users.me
            if user is not None:
                return gis
            print("Warning: ArcGIS Pro is connected, but no AGOL user was found.")
        except Exception as error:
            print(f"Could not connect to ArcGIS Pro credentials: {error}")

    print("Using environment variables or interactive prompt...")
    portal = os.getenv("AGOL_URL", "https://www.arcgis.com")
    username = os.getenv("AGOL_USERNAME") or input("AGOL username: ")
    password = os.getenv("AGOL_PASSWORD") or getpass.getpass("AGOL password: ")
    return GIS(portal, username, password)


gis = get_agol_connection(use_pro=False)  # Set to True to try ArcGIS Pro first
user = gis.users.me
if user is None:
    raise RuntimeError("Login failed: AGOL user information is unavailable.")

print(f">>> Logged in as: {user.username}")
print(
    f">>> AGOL organization / Portal: {gis.properties.get('urlKey')} ({gis.properties.get('name')})"
)

In [ ]:
# --- Function: Load a specific Python module from GitHub ---
def load_github_module(
    module: str,
    owner: str,
    repo: str,
    reference: str | None = None,
):
    """
    Load one Python module file from a GitHub repository into the current AGOL session.

    `reference` can be a branch, tag, or commit hash.

    :param module: Path to module file in repository, for example "modules/hello_world.py".
    :param owner: GitHub owner or organization.
    :param repo: GitHub repository name.
    :param reference: Branch, tag, or commit hash. Defaults to "main".
    :return: Imported module object.
    """

    import importlib.util
    import urllib.request

    reference = reference or "main"
    local_file = f"/arcgis/home/modules_cache/{reference[:7]}/{module}"
    os.makedirs(os.path.dirname(local_file), exist_ok=True)

    url = f"https://raw.githubusercontent.com/{owner}/{repo}/{reference}/{module}"
    if not os.path.exists(local_file):
        urllib.request.urlretrieve(url, local_file)

    module_stub = module.removesuffix(".py")
    module_name = f"{module_stub}_{reference[:7]}"

    spec = importlib.util.spec_from_file_location(module_name, local_file)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not create import spec for {local_file}")

    loaded_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(loaded_module)
    return loaded_module

### Example A: load module `agol_user_admin.py` by version tag from `arcgis_utils`

- reproducible
- recommended for stable releases


In [ ]:
# Load module `agol_user_admin.py` by version tag from the arcgis_utils subpackage
agol_user_admin = load_github_module(
    owner="miljodirektoratet",
    repo="gis-utils-public",
    reference="v0.0.6",
    module="src/gis_utils_public/arcgis_utils/agol_user_admin.py",
)

# Load module `main.py` by version tag from the main package gis_utils_public
main = load_github_module(
    owner="miljodirektoratet",
    repo="gis-utils-public",
    reference="v0.0.6",
    module="src/gis_utils_public/main.py",
)
main.main()

### Example B: load module `main.py` by commit hash

- reproducible
- can be pointed to an exact file version

In [ ]:
# Load module `main.py` by commit hash  from the main package gis_utils_public
main = load_github_module(
    owner="miljodirektoratet",
    repo="gis-utils-public",
    reference="163e0cea6c0178b6483da7bd8ce63bf75adb1422",
    module="src/gis_utils_public/main.py",
)

main.main()

### Example C: load module `main.py` by branch name

- not-reproducible
- only use in developement phase, do not use for scheduled notebooks in production

In [ ]:
# Load module `main.py` by branch name from the main package gis_utils_public
main = load_github_module(
    owner="miljodirektoratet",
    repo="gis-utils-public",
    reference="main",
    module="src/gis_utils_public/main.py",
)

main.main()

### Example D: install the package `gis_utils_public` on AGOL

- not recommended as this will increase credit costs! 
- do not use this pattern instead choose one of the module import examples above

In [ ]:
# Install the package `gis_utils_public` (not recommended, costs more credits)
# Uncomment the line below to install, note this costs more credits than loading specific modules
%pip install --no-cache-dir "git+https://github.com/miljodirektoratet/gis-utils-public.git@v0.0.6"

In [ ]:
# Check what's in the subpackage `arcgis_utils`
print(f"__all__: {arcgis_utils.__all__}")

# Use a function from the `arcgis_utils` subpackage to verify it works
# <subpackage>.<module>.<function>()
# arcgis_utils.agol_user_admin.agol_add_users_to_group()